# 05 · 企业并购估值：DCF、协同效应定价与商誉

站在**收购方**的角度，回答投决会上最常被问的四个问题：

1. 标的单独值多少钱？（DCF 估值）
2. 这个估值有多可靠？（敏感性分析）
3. 我们最多能出多少？溢价给多了会怎样？（协同效应定价）
4. 买贵了，账上会出现什么？（商誉与减值）

所有参数都是**教学用的示例值**（单位：百万元），实际使用时换成标的的真实数据，以及当期的无风险利率和风险溢价（例如 Damodaran 每年更新的数据页）。无需联网。

> 本仓库仅供学习，所有内容不构成投资建议。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from capital_budgeting import npv   # 复用资本预算脚本里的净现值函数

pd.set_option("display.float_format", "{:,.1f}".format)

## 1. 资本成本（WACC）

- 股权成本用 CAPM：`ke = rf + β × ERP`
- 债务成本取税后：`kd × (1 − 税率)`
- 按目标资本结构加权：`WACC = E/V × ke + D/V × kd × (1 − t)`

注意：要用**标的业务**的风险，而不是收购方自己的 WACC。用低风险母公司的资本成本去评估高风险标的，会系统性地高估标的价值。

In [ ]:
rf, beta, erp = 0.020, 1.10, 0.060      # 无风险利率、标的行业 β、股权风险溢价（示例值）
kd, tax = 0.040, 0.25                    # 税前债务成本、所得税率
d_v = 0.30                               # 目标资本结构中债务占比

ke = rf + beta * erp
wacc = (1 - d_v) * ke + d_v * kd * (1 - tax)
print(f"股权成本 ke = {ke:.2%}")
print(f"WACC       = {wacc:.2%}")

## 2. 标的单独估值：五年预测 + 永续期

自由现金流（FCFF）= EBIT × (1 − 税率) + 折旧 − 资本开支 − 营运资金增加。永续期用戈登增长模型：`TV = FCF_{n+1} / (WACC − g)`，要求 g 小于 WACC，且不应高于长期名义经济增速。

In [ ]:
years = np.arange(1, 6)
revenue = 800 * np.cumprod([1.12, 1.10, 1.08, 1.06, 1.05])     # 收入增速逐年放缓
ebit = revenue * 0.15
da = revenue * 0.04                                              # 折旧摊销
capex = revenue * 0.05
d_nwc = np.diff(np.concatenate([[800 * 0.10], revenue * 0.10]))  # 营运资金占收入 10%
fcff = ebit * (1 - tax) + da - capex - d_nwc

forecast = pd.DataFrame({"revenue": revenue, "EBIT": ebit, "D&A": da, "capex": capex,
                         "ΔNWC": d_nwc, "FCFF": fcff}, index=[f"Y{y}" for y in years])
forecast

In [ ]:
def dcf_value(fcff, wacc, g):
    pv_explicit = npv(wacc, [0.0, *fcff])                       # 第 1–5 年
    tv = fcff[-1] * (1 + g) / (wacc - g)
    pv_tv = tv / (1 + wacc) ** len(fcff)
    return pv_explicit + pv_tv, pv_explicit, pv_tv


g = 0.025
ev, pv_explicit, pv_tv = dcf_value(fcff, wacc, g)
net_debt = 300
equity = ev - net_debt
print(f"企业价值 EV       = {ev:,.1f}")
print(f"  其中预测期现值   = {pv_explicit:,.1f}")
print(f"  其中终值现值     = {pv_tv:,.1f}（占 {pv_tv / ev:.0%}）")
print(f"股权价值 = EV − 净负债 {net_debt} = {equity:,.1f}")

**终值通常占企业价值的大半**。这意味着估值高度依赖“五年以后会怎样”的假设，所以下一步的敏感性分析不是可选项，而是投决会材料的必备内容。

## 3. 敏感性：WACC × 永续增长率

In [ ]:
waccs = [wacc - 0.01, wacc - 0.005, wacc, wacc + 0.005, wacc + 0.01]
gs = [0.015, 0.020, 0.025, 0.030, 0.035]
grid = pd.DataFrame([[dcf_value(fcff, w, gg)[0] - net_debt for gg in gs] for w in waccs],
                    index=[f"WACC {w:.2%}" for w in waccs], columns=[f"g {gg:.1%}" for gg in gs])
grid

In [ ]:
lo, hi = grid.values.min(), grid.values.max()
print(f"股权价值区间：{lo:,.0f} ~ {hi:,.0f}，最高约为最低的 {hi / lo:.1f} 倍")

## 4. 协同效应定价：最高能出多少

收购方能接受的**最高价格** = 标的单独价值 + 协同效应现值 − 整合成本现值。

- 成本协同通常比收入协同更可靠，也更快兑现；收入协同要打更大的折扣。
- 协同不会第一年就全部实现，要按爬坡节奏估算。
- 整合本身有成本（人员安置、系统切换、咨询费等）。

In [ ]:
full_synergy = 60                                    # 每年税前成本协同（完全实现后）
ramp = np.array([0.3, 0.7, 1.0, 1.0, 1.0])           # 实现节奏
synergy_after_tax = full_synergy * ramp * (1 - tax)
integration_cost = np.array([-80, -40, 0, 0, 0]) * (1 - tax)   # 一次性整合成本（税后）
net_synergy = synergy_after_tax + integration_cost

pv_synergy = npv(wacc, [0.0, *net_synergy]) + (synergy_after_tax[-1] / wacc) / (1 + wacc) ** 5
max_equity_price = equity + pv_synergy
print(f"标的单独股权价值     = {equity:,.1f}")
print(f"协同效应现值（净）   = {pv_synergy:,.1f}（永续期假设协同不再增长）")
print(f"最高可接受股权价格   = {max_equity_price:,.1f}")

In [ ]:
rows = []
for premium in [0.0, 0.2, 0.4, 0.6]:
    price = equity * (1 + premium)
    value_to_buyer = max_equity_price - price
    rows.append({"收购溢价": f"{premium:.0%}", "收购价": price,
                 "买方获得的价值": value_to_buyer,
                 "协同效应中付给卖方的比例": f"{(price - equity) / pv_synergy:.0%}"})
pd.DataFrame(rows).set_index("收购溢价")

溢价越高，协同效应中付给卖方的部分越多；当溢价超过协同效应现值时，**即使协同全部兑现，这笔交易也在毁灭收购方股东的价值**。投决会上要明确：协同效应中最多愿意让渡多少给卖方。

## 5. 商誉：买贵了，账上会出现什么

按《企业会计准则第 20 号——企业合并》，非同一控制下的合并中，**合并成本超过取得的可辨认净资产公允价值份额的部分确认为商誉**；按《企业会计准则第 8 号——资产减值》，商誉**至少每年年末做减值测试**。

下面假设以 40% 的溢价收购 100% 股权，一年后发现业绩只能达到原预测的 75%，并且判断以后各年也只能维持在这个水平。

In [ ]:
price_paid = equity * 1.40
identifiable_net_assets_fv = 600            # 可辨认净资产公允价值（示例值）
goodwill = price_paid - identifiable_net_assets_fv
print(f"收购价 {price_paid:,.1f} − 可辨认净资产公允价值 {identifiable_net_assets_fv} = 商誉 {goodwill:,.1f}")

# 一年后：业绩只能达到原预测的 75%（且判断会持续），用同样的方法重估资产组可收回金额
actual_fcff = fcff * 0.75
recoverable_ev, _, _ = dcf_value(actual_fcff, wacc, g)
recoverable_equity = recoverable_ev - net_debt
carrying = identifiable_net_assets_fv + goodwill                 # 简化：资产组账面价值
impairment = max(carrying - recoverable_equity, 0)
impairment = min(impairment, goodwill)                           # 先冲减商誉
print(f"可收回金额 {recoverable_equity:,.1f} vs 账面价值 {carrying:,.1f}")
print(f"需计提商誉减值 {impairment:,.1f}，占商誉的 {impairment / goodwill:.0%}")

这正是很多并购的典型路径：**高溢价 → 业绩承诺期内勉强达标 → 承诺期结束后业绩回落 → 商誉大额减值，一次性冲击利润**。所以投决时就要问：可辨认净资产之外，我们为什么愿意多付这么多？这部分价值靠什么兑现？兑现不了时谁承担？

## 练习

1. 把 β 从 1.1 改成 1.4（更高风险的业务），估值下降多少？如果误用了收购方自己 0.8 的 β，会高估多少？
2. 把收入协同（每年 40，按 50% 折扣）加进协同效应，最高可接受价格怎么变？
3. 设计一个业绩承诺 + 分期付款的交易结构，让一部分对价在协同兑现后才支付，并重新计算买方承担的风险。
4. 对照 `corporate-investment/04-ma-and-integration.md`，为这笔交易写一份投决会报告（模板：`corporate-investment/templates/ic-memo.md`）。